In [1]:
from google.colab import userdata
import os

os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

In [2]:
import os, sys

REPO_ROOT = "/content/dispersion-validity-mask-fix-4staso"   # adjust if different

if not os.path.exists(REPO_ROOT):
    # Replace with your actual GitHub URL
    os.system(f"git clone https://$GITHUB_TOKEN@github.com/lucastiger/tfln-soliton-control.git {REPO_ROOT}")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

In [3]:
#!/usr/bin/env python3
"""Deterministic reference: clean single-DKS spectrum with measured dispersion.

Reproduces (numpy, no JAX, no RNG) the reference figure panel
"n=16384 dealiased, corrected-gauge DW check" from the dispersion audit:
smooth sech^2 comb, symmetric tails, DW peaks at ~1095 nm / ~2529 nm, and a
machine-zero floor beyond the 2/3 cutoff. This is the ground truth the
pipeline spectrum must match qualitatively.

Numerical scheme = the repo's legacy round-trip map (pump kick, Strang
L/2 - NL - L/2 at dt = t_r) + a hard 2/3 dealias mask folded into the linear
half-step. NO edge absorber, NO dispersion-validity mask, NO thermal loop,
NO detuning noise. complex128 throughout.

Run:  python reference_soliton_spectrum.py [--dw-kappa 8.0] [--steps 12000]
Deterministic: identical output every run.
"""
import argparse
import numpy as np
import sys
import os # Import os module to handle paths

# ---------------- fixed physical constants (repo config values) -------------
C0 = 299792458.0
KAPPA = 1.519e8            # rad/s   total linewidth
KAPPA_C = 1.215e8          # rad/s   coupling rate
GAMMA = 1.029e18           # 1/(J s) Kerr coefficient (per-photon-energy units)
PIN = 0.214                # W       pump power
TR = 1.0 / 2.46e10         # s       round-trip time (config FSR)
N_TAU = 16384              # modes; 2/3 cutoff |mu| = 5461 keeps both DW bands

# Construct the absolute path to the CSV file using REPO_ROOT
# REPO_ROOT is defined in a previous cell (2FxarVy208qg)
# Assuming REPO_ROOT is already defined and available in the environment.
# If not, it would need to be retrieved or redefined here.
REPO_ROOT = "/content/dispersion-validity-mask-fix-4staso"
CSV = os.path.join(REPO_ROOT, "config/pyLLE_dispersion_w4400_h800.csv")


def build_grid(n):
    """Rest-gauge D_int on the FFT bin grid (matches post-audit loader)."""
    data = np.loadtxt(CSV, delimiter=",")
    mu = data[:, 0].astype(np.int64)
    omega = 2.0 * np.pi * data[:, 1]
    i0 = int(np.where(mu == 0)[0][0])
    sel = (np.abs(mu) <= 600) & (np.abs(mu) > 5)          # exclude pump defect
    pf = np.polynomial.Polynomial.fit(mu[sel].astype(float), omega[sel], 7)
    d1 = float(pf.deriv()(0.0))                           # wide-fit D1
    dint = omega - omega[i0] - d1 * mu                    # D_int(0)=0 exact
    fsr = d1 / (2.0 * np.pi)
    k = np.round(np.fft.fftfreq(n, d=(1.0 / fsr) / n) / fsr).astype(int)
    grid = np.interp(k, mu, dint)                         # slope-clamped ends:
    lo, hi = k < mu.min(), k > mu.max()                   # continue edge slope
    grid[lo] = dint[0] + (dint[1] - dint[0]) * (k[lo] - mu.min())
    grid[hi] = dint[-1] + (dint[-1] - dint[-2]) * (k[hi] - mu.max())
    return grid, k, fsr, data[i0, 1]                      # grid, bins, fsr, f0


def run(dw_kappa=8.0, steps=12000, n=N_TAU):
    dw = dw_kappa * KAPPA
    grid, k, fsr, f0 = build_grid(n)
    beta2 = 9.89401e4 / (2.0 * np.pi * fsr) ** 2          # seed width only
    tau_s = np.sqrt(beta2 / (2.0 * dw))
    t = np.arange(n) * TR / n
    e = (np.sqrt(KAPPA_C * PIN) / (KAPPA / 2 + 1j * dw)
         + np.sqrt(2 * dw / GAMMA) / np.cosh((t - t[n // 2]) / tau_s)
         ).astype(np.complex128)
    F = np.sqrt(KAPPA_C * PIN) * TR
    mask = (np.abs(k) <= n / 3.0).astype(float)           # hard 2/3 dealias
    h = np.exp((-KAPPA / 2 - 1j * grid - 1j * dw) * TR / 2) * mask
    for _ in range(steps):
        e = e + F
        e = np.fft.ifft(np.fft.fft(e) * h)
        e = e * np.exp(1j * GAMMA * np.abs(e) ** 2 * TR)
        e = np.fft.ifft(np.fft.fft(e) * h)
    s = np.abs(np.fft.fftshift(np.fft.fft(e))) ** 2
    sdb = 10 * np.log10(np.maximum(s / s.max(), 1e-36))
    mus = np.arange(n) - n // 2
    return mus, sdb, grid, fsr, f0, dw


def validate(mus, sdb, grid, fsr, f0, dw, n=N_TAU):
    """Objective spectral-integrity checks (V1-V5). Raises on failure."""
    rep = {}
    # V1 tail linearity + symmetry: linear dB fit over 400<=|mu|<=1500
    slopes = {}
    for sgn in (+1, -1):
        sel = (sgn * mus >= 400) & (sgn * mus <= 1500)
        p = np.polyfit(np.abs(mus[sel]), sdb[sel], 1)
        res = np.std(sdb[sel] - np.polyval(p, np.abs(mus[sel])))
        assert res < 3.0, f"V1 tail not smooth (RMS {res:.1f} dB, sign {sgn})"
        slopes[sgn] = p
    asym = abs(slopes[1][0] - slopes[-1][0]) / abs(slopes[1][0])
    assert asym < 0.15, f"V1 tail slope asymmetry {asym:.2f} > 0.15"
    rep["tail_slopes_dB_per_mode"] = (slopes[1][0], slopes[-1][0])
    # V2 no interior plateau/hole between comb and DWs (the validity-mask bug)
    dwx = [m for m in _crossings(grid, dw, n) if abs(m) > 500]
    for sgn in (+1, -1):
        band = (sgn * mus > 300) & (sgn * mus < 2900)
        for m in dwx:
            band &= np.abs(mus - m) > 80
        ref = np.polyval(slopes[sgn], np.abs(mus[band]))
        worst = np.min(sdb[band] - ref)
        assert worst > -25.0, (
            f"V2 interior hole: {worst:.0f} dB below tail on sign {sgn} "
            f"(validity-mask-type amputation)")
    # V3 sech^2 core
    # dB-space fit is the robust test: linear-power R^2 is dominated by the
    # few strongest lines + CW-background interference at small mu.
    core = (np.abs(mus) <= 200) & (np.abs(mus) >= 3)  # exclude pump line
    from scipy.optimize import curve_fit
    fit = lambda m, a, mc: a - 20 * np.log10(
        np.cosh(np.pi * m / (2 * np.abs(mc))))
    (a, mc), _ = curve_fit(fit, mus[core], sdb[core], p0=(-20.0, 300.0))
    rms = np.std(sdb[core] - fit(mus[core], a, mc))
    assert rms < 2.0, f"V3 core not sech^2 (dB RMS {rms:.2f} > 2)"
    rep["core_mu_c"] = abs(mc)
    rep["core_fit_rms_dB"] = round(float(rms), 2)
    # V4 DW positions and prominence
    for m in dwx:
        w = np.abs(mus - m) < 60
        pk, mp = sdb[w].max(), mus[w][np.argmax(sdb[w])]
        prom = pk - np.polyval(slopes[np.sign(m)], abs(mp))
        assert prom > 50, f"V4 DW at mu={m} prominence {prom:.0f} dB < 50"
        rep[f"DW_mu{m:+d}"] = (int(mp), round(pk, 1),
                               round(C0 / (f0 + mp * fsr) * 1e9))
    # V5 machine floor beyond 2/3 cutoff
    out = np.abs(mus) > n / 3 + 50
    assert np.median(sdb[out]) < -300, "V5 dealiased floor above -300 dB"
    return rep


def _crossings(grid, dw, n):
    g = np.fft.fftshift(grid)
    sgn = np.sign(g - dw)
    return [int(i - n // 2) for i in np.where(np.diff(sgn) != 0)[0]]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--dw-kappa", type=float, default=8.0)
    ap.add_argument("--steps", type=int, default=12000)

    # Check if running in a Jupyter/IPython environment
    if 'ipykernel' in sys.modules:
        args = ap.parse_args([]) # Pass an empty list to avoid parsing kernel arguments
    else:
        args = ap.parse_args() # Parse actual command-line arguments when run as a script

    mus, sdb, grid, fsr, f0, dw = run(args.dw_kappa, args.steps)
    rep = validate(mus, sdb, grid, fsr, f0, dw)
    print("ALL CHECKS PASSED:", rep)
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        lam = C0 / (f0 + mus * fsr) * 1e9
        fig, ax = plt.subplots(figsize=(10, 4.5))
        ax.plot(lam, sdb, lw=0.4)
        ax.set(xlim=(950, 2700), ylim=(-200, 3), xlabel="wavelength (nm)",
               ylabel="dB rel. max",
               title=f"reference DKS spectrum, delta_omega={args.dw_kappa}k")
        fig.tight_layout()
        fig.savefig("reference_soliton_spectrum.png", dpi=130)
        print("wrote reference_soliton_spectrum.png")
    except ImportError:
        pass

ALL CHECKS PASSED: {'tail_slopes_dB_per_mode': (np.float64(-0.03968137733521701), np.float64(-0.04221679048897625)), 'core_mu_c': np.float64(314.1776442592144), 'core_fit_rms_dB': 0.76, 'DW_mu-3052': (-3069, np.float64(-93.0), 2529), 'DW_mu+3270': (3281, np.float64(-95.4), 1095)}
wrote reference_soliton_spectrum.png
